# Experiment Results Visualization
Loads pre-computed JSON/CSV result files and renders:
- Table 1: Main comparison (deep / RF / stack vs baselines)
- Table 2: Ablation study
- Figure 1: Regime membership radar
- Figure 2: Expert weight heatmap
- Figure 3: Case study (prediction vs ground-truth)
- Figure 4: Soft-gate entropy vs RMSE

In [ ]:
import sys
sys.path.insert(0, '..')

import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
RESULTS = Path('../results')
INTERPRET = RESULTS / 'interpret'
STATIONS = [f'station{i:02d}' for i in range(10)]

## Table 1 — Main results (deep / RF / stack)

In [ ]:
def load_stack_results(results_dir=RESULTS):
    rows = []
    for p in sorted(results_dir.glob('stack_*.json')):
        with open(p) as f:
            rows.append(json.load(f))
    return rows

stack_rows = load_stack_results()
methods = ['deep', 'rf', 'stack']
table1_rows = []
for m in methods:
    vals = pd.DataFrame([r[m] for r in stack_rows])
    table1_rows.append({
        'Method': {'deep': 'MoE-Deep', 'rf': 'MoE-RF', 'stack': 'MoE-Stack'}[m],
        'ACC (%)': f"{vals['acc'].mean():.2f} ± {vals['acc'].std():.2f}",
        'RMSE': f"{vals['rmse'].mean():.4f} ± {vals['rmse'].std():.4f}",
        'MAE': f"{vals['mae'].mean():.4f} ± {vals['mae'].std():.4f}",
        'R²': f"{vals['r2'].mean():.4f} ± {vals['r2'].std():.4f}",
        'QR (%)': f"{vals['qr'].mean():.2f} ± {vals['qr'].std():.2f}",
    })
table1 = pd.DataFrame(table1_rows).set_index('Method')
print('=== Table 1: Main results (10 stations × 5 seeds) ===')
table1

In [ ]:
# Per-station ACC bar chart
per_station = {m: {} for m in methods}
for r in stack_rows:
    sid = r['station']
    for m in methods:
        per_station[m].setdefault(sid, []).append(r[m]['acc'])

x = np.arange(len(STATIONS))
width = 0.25
colors = ['#4878CF', '#6ACC65', '#D65F5F']
labels = ['MoE-Deep', 'MoE-RF', 'MoE-Stack']

fig, ax = plt.subplots(figsize=(14, 5))
for i, (m, color, label) in enumerate(zip(methods, colors, labels)):
    means = [np.mean(per_station[m].get(s, [np.nan])) for s in STATIONS]
    stds  = [np.std(per_station[m].get(s, [np.nan]))  for s in STATIONS]
    ax.bar(x + i * width, means, width, yerr=stds, label=label,
           color=color, capsize=3, alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels([s[-2:] for s in STATIONS])
ax.set_xlabel('Station')
ax.set_ylabel('ACC (%)')
ax.set_title('Per-station ACC: Deep / RF / Stack')
ax.legend()
ax.set_ylim(80, 100)
plt.tight_layout()
plt.savefig('../results/fig_main_acc.pdf', bbox_inches='tight')
plt.show()

## Table 1b — Baseline comparison

In [ ]:
bl_csv = RESULTS / 'baselines_table.csv'
if bl_csv.exists():
    bl = pd.read_csv(bl_csv, index_col=0)
    print('=== Baseline ACC (mean ± std per station) ===')
    display(bl)
else:
    print(f'{bl_csv} not found — run scripts/train_baselines.sh first')

## Table 2 — Ablation study

In [ ]:
abl_csv = RESULTS / 'ablation' / 'summary.csv'
if not abl_csv.exists():
    print(f'{abl_csv} not found — run scripts/ablation.sh first')
else:
    abl = pd.read_csv(abl_csv)
    VARIANT_LABEL = {
        'A': 'A  Full model',
        'B': 'B  Hard gating',
        'C': 'C  Fixed equal gate',
        'D': 'D  Homogeneous experts',
        'E': 'E  Single expert',
        'G2': 'G2 K=2',
        'G4': 'G4 K=4',
        'G5': 'G5 K=5',
        'H': 'H  VMD leakage (oracle)',
        'I': 'I  Irradiance anchor',
    }
    abl['label'] = abl['variant'].map(VARIANT_LABEL).fillna(abl['variant'])
    baseline_acc = float(abl[abl['variant'] == 'A']['acc_mean'].values[0])
    abl['Δ ACC'] = abl['acc_mean'] - baseline_acc
    display_cols = ['label', 'acc_mean', 'acc_std', 'rmse_mean', 'mae_mean', 'r2_mean', 'Δ ACC']
    out = abl[display_cols].rename(columns={
        'label': 'Variant', 'acc_mean': 'ACC mean', 'acc_std': 'ACC std',
        'rmse_mean': 'RMSE', 'mae_mean': 'MAE', 'r2_mean': 'R²'
    })
    print('=== Table 2: Ablation study ===')
    display(out.style.format({
        'ACC mean': '{:.2f}', 'ACC std': '{:.2f}',
        'RMSE': '{:.4f}', 'MAE': '{:.4f}', 'R²': '{:.4f}', 'Δ ACC': '{:+.2f}'
    }))

## Figure 1 — Weather regime membership radar
Requires `python -m src.run interpret` to generate `regime_membership_{sid}.csv`.

In [ ]:
sid = 'station00'
rm_path = INTERPRET / f'regime_membership_{sid}.csv'

if not rm_path.exists():
    print(f'{rm_path} not found — run: python -m src.run interpret --station {sid}')
else:
    rm = pd.read_csv(rm_path)  # columns: regime_0, regime_1, ...
    regime_cols = [c for c in rm.columns if c.startswith('regime_')]
    K = len(regime_cols)
    U = rm[regime_cols].values  # (n_days, K)

    if K == 3:
        fig, ax = plt.subplots(figsize=(6, 6))
        def to_cart(u):
            x = 0.5 * (2 * u[:, 1] + u[:, 2]) / u.sum(1)
            y = (np.sqrt(3) / 2) * u[:, 2] / u.sum(1)
            return x, y
        x, y = to_cart(U)
        sc = ax.scatter(x, y, c=U[:, 0], cmap='RdYlBu', s=8, alpha=0.5, vmin=0, vmax=1)
        plt.colorbar(sc, ax=ax, label='regime_0 membership')
        ax.set_aspect('equal')
        ax.set_title(f'{sid} — FCM regime membership (ternary, K=3)')
        ax.axis('off')
        tri = plt.Polygon([[0, 0], [1, 0], [0.5, np.sqrt(3)/2]], fill=False, edgecolor='k')
        ax.add_patch(tri)
        ax.text(-0.05, -0.05, 'Regime 0', ha='center')
        ax.text(1.05, -0.05, 'Regime 1', ha='center')
        ax.text(0.5, np.sqrt(3)/2 + 0.05, 'Regime 2', ha='center')
        plt.tight_layout()
        plt.savefig('../results/fig1_regime_ternary.pdf', bbox_inches='tight')
        plt.show()
    else:
        means = U.mean(axis=0)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar(range(K), means, color='steelblue')
        ax.set_xlabel('Regime')
        ax.set_ylabel('Mean membership')
        ax.set_title(f'{sid} — Mean FCM regime membership')
        plt.tight_layout()
        plt.savefig('../results/fig1_regime_bar.pdf', bbox_inches='tight')
        plt.show()

## Figure 2 — Expert weight heatmap
Requires `expert_weight_{sid}.csv` from `src.run interpret`.

In [ ]:
sid = 'station00'
ew_path = INTERPRET / f'expert_weight_{sid}.csv'

if not ew_path.exists():
    print(f'{ew_path} not found — run: python -m src.run interpret --station {sid}')
else:
    ew = pd.read_csv(ew_path)  # columns: expert_0, expert_1, ...
    expert_cols = [c for c in ew.columns if c.startswith('expert_')]
    K = len(expert_cols)

    # 按dominant regime分组（从regime_membership取）
    rm = pd.read_csv(INTERPRET / f'regime_membership_{sid}.csv')
    regime_cols = [c for c in rm.columns if c.startswith('regime_')]
    dominant = rm[regime_cols].values.argmax(axis=1)
    ew['dominant_regime'] = dominant[:len(ew)]

    pivot = ew.groupby('dominant_regime')[expert_cols].mean()
    pivot.columns = [f'Expert {k}' for k in range(K)]
    pivot.index = [f'Regime {r}' for r in pivot.index]

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd',
                vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title(f'{sid} — Mean expert gate weights per regime')
    plt.tight_layout()
    plt.savefig('../results/fig2_expert_heatmap.pdf', bbox_inches='tight')
    plt.show()

## Figure 3 — Case study: prediction vs ground-truth

In [ ]:
import re

sid = 'station00'
seed = 0
# Find predictions file (may have timestamp suffix from old runs or no suffix)
pred_candidates = sorted((RESULTS / 'ablation' / 'A' / 'eval').glob(f'predictions_{sid}*.csv'))
if not pred_candidates:
    # Try results root
    pred_candidates = sorted(RESULTS.glob(f'predictions_{sid}*seed{seed}*.csv'))

if not pred_candidates:
    print('No predictions CSV found. Run: python -m src.run evaluate --station', sid)
else:
    pred = pd.read_csv(pred_candidates[0])
    print(f'Loaded {pred_candidates[0].name}, shape={pred.shape}')
    print(pred.columns.tolist())

    # Assume columns: y_true, y_pred (or similar)
    true_col = [c for c in pred.columns if 'true' in c.lower() or c == 'y'][0]
    pred_col = [c for c in pred.columns if 'pred' in c.lower() or 'hat' in c.lower()][0]

    # Pick a 3-day window (index 0..287)
    n_plot = 96 * 3
    t = np.arange(n_plot) * 15 / 60  # hours

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, pred[true_col].values[:n_plot], label='Ground truth', lw=1.5, color='#2c7bb6')
    ax.plot(t, pred[pred_col].values[:n_plot], label='Prediction',   lw=1.5, color='#d7191c', alpha=0.85)
    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Power')
    ax.set_title(f'{sid} seed={seed} — 3-day prediction case study')
    ax.legend()
    for d in range(1, 3):
        ax.axvline(d * 24, color='gray', lw=0.8, ls='--')
    plt.tight_layout()
    plt.savefig('../results/fig3_case_study.pdf', bbox_inches='tight')
    plt.show()

## Figure 4 — Soft-gate entropy vs RMSE
Requires `entropy_error_{sid}.csv` from `src.run interpret`.

In [ ]:
sid = 'station00'
ee_path = INTERPRET / f'entropy_error_{sid}.csv'

if not ee_path.exists():
    print(f'{ee_path} not found — run: python -m src.run interpret --station {sid}')
else:
    ee = pd.read_csv(ee_path)  # columns: entropy, rmse_day, dominant_regime

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(ee['entropy'], ee['rmse_day'], s=15, alpha=0.5, color='steelblue')

    coef = np.polyfit(ee['entropy'], ee['rmse_day'], 1)
    x_line = np.linspace(ee['entropy'].min(), ee['entropy'].max(), 100)
    ax.plot(x_line, np.polyval(coef, x_line), color='crimson', lw=1.5, label='linear fit')

    corr = ee['entropy'].corr(ee['rmse_day'])
    ax.set_xlabel('Gate entropy (bits)')
    ax.set_ylabel('Daily RMSE')
    ax.set_title(f'{sid} — Gate entropy vs daily RMSE  (r={corr:.3f})')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/fig4_entropy_rmse.pdf', bbox_inches='tight')
    plt.show()

## Appendix — Per-station detailed metrics

In [ ]:
per_sid_rows = []
for r in stack_rows:
    sid = r['station']
    for m in methods:
        v = r[m]
        per_sid_rows.append({
            'station': sid, 'method': m, 'seed': r['seed'],
            'acc': v['acc'], 'rmse': v['rmse'], 'mae': v['mae'], 'r2': v['r2']
        })
per_sid = pd.DataFrame(per_sid_rows)
agg = per_sid.groupby(['station', 'method']).agg(
    acc_mean=('acc', 'mean'), acc_std=('acc', 'std'),
    rmse_mean=('rmse', 'mean')
).round(3)
display(agg)